### ETL: bronze.events -> silver.events_cleaned

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types as T import  FloatType, DateType, StringType IntegerType
from delta.tables import DeltaTable 
import sys
import os
from pathlib import Path
current_dir = "/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
src_path = str(Path(current_dir).parents[1])
sys.path.append(src_path)

from utils.cleaning_functions import *

In [0]:
df_source = spark.table("dbw_routemind_euskadi_dev.bronze.events")

In [0]:
df_exploded = explode_array_column(df_source,"items","item")

In [0]:
cast_map = {
    "id": T.StringType(),
    "nameEs": T.StringType(),
    "type": T.IntegerType(),
    "typeEs": T.StringType(),
    "placeEs": T.StringType(),
    "municipalityEs": T.StringType(),
    "provinceNoraCode": T.IntegerType(),
    "endDate": T.DateType(),
    "startDate": T.DateType(),
    "openingHoursEs": T.StringType()
}

notnull_columns = [
    "id",
    "type",
    "typeEs",
    "municipalityEs",
    "provinceNoraCode",
    "startDate",
    "endDate"
]

keys=["id"]


In [0]:
df_extracted = extract_and_cast(df_exploded,"item",cast_map)

In [0]:
df_clean = drop_null_required(df_extracted, notnull_columns)

In [0]:
df_final = deduplicate(df_clean, keys)

In [0]:
df_final = df_final.select(
    col("id").alias("id"),
    col("nameEs").alias("name"),
    col("type").alias("type_id"),
    col("typeEs").alias("type_name"),
    col("placeEs").alias("location")
    col("municipalityEs").alias("municipality"),
    col("provinceNoraCode").alias("county_id"),
    col("startDate")
    Col("endDate")
    col("openingHoursEs").alias("opening_hour")
)

In [0]:
target_table = "dbw_routemind_euskadi_dev.silver.events_cleaned"
delta_path = "abfss://silver@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/events/data"

df_clean.write \
    .format("delta") \
    .option("path", delta_path) \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(target_table)


print(f"APPEND completed on {target_table}. rows processed: {df_clean.count()}")